In [ ]:
!pip install noisereduce
import os, random, warnings
import numpy as np
import librosa
import noisereduce as nr
from scipy import signal
warnings.filterwarnings("ignore")
import os
import cv2  # <-- The thread-safe hero
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split


# ── CONFIG ─────────────────────────────────────────────────────────────────────
TARGET_SR  = 48000
TRIM_TOP_DB   = 40     
HP_CUTOFF     = 50     
HP_ORDER      = 4      
NOISE_PROP    = 0.75   
TARGET_RMS    = 0.05
BASE_PATH = "/kaggle/input/datasets/alieldinalaa/nn-cmp27-dataset" 
TARGET_SR = 48000
LABEL_MAP = {
    "Machine 1_Normal": 0, "Machine 1_Abnormal": 1,
    "Machine 2_Normal": 2, "Machine 2_Abnormal": 3,
    "Machine 3_Normal": 4, "Machine 3_Abnormal": 5
}
SAVE_PATH = '/kaggle/working/cleaned_melspecs_checkpoint.npz'

In [ ]:
# ── CLEANING FUNCTIONS ─────────────────────────────────────────────────────────
def _find_quietest_clip(y, sr, clip_dur=0.3):
    frame_len = int(sr * clip_dur)
    hop       = frame_len // 2
    if len(y) <= frame_len: return y
    rms_vals   = [np.sqrt(np.mean(y[i:i+frame_len]**2)) for i in range(0, len(y)-frame_len, hop)]
    best_start = int(np.argmin(rms_vals)) * hop
    return y[best_start : best_start + frame_len]

def clean_audio(y, sr, trim_top_db=TRIM_TOP_DB, hp_cutoff=HP_CUTOFF, hp_order=HP_ORDER, noise_prop=NOISE_PROP, target_rms=TARGET_RMS):
    y_trim, _ = librosa.effects.trim(y, top_db=trim_top_db, frame_length=2048, hop_length=512)
    noise_clip = _find_quietest_clip(y_trim, sr)
    y_nr = nr.reduce_noise(y=y_trim, sr=sr, y_noise=noise_clip, stationary=True, prop_decrease=noise_prop)
    rms = np.sqrt(np.mean(y_nr**2))
    if rms > 1e-9: y_nr = y_nr * (target_rms / rms)
    sos  = signal.butter(hp_order, hp_cutoff, btype="hp", fs=sr, output="sos")
    y_hp = signal.sosfilt(sos, y_nr)
    return np.clip(y_hp, -1.0, 1.0)

def extract_standardized_mel(file_info):
    machine, state, file_path = file_info
    try:
        # Strictly 10 seconds. No trimming.
        y, sr = librosa.load(file_path, sr=TARGET_SR, duration=10.0)
        if len(y) < 1024: return None

        # Inject the cleaning pipeline
        y = clean_audio(y, sr)
        if len(y) < 1024: return None
        
        # 1. Mel-Spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        
        # 2. Convert to Decibels
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # 3. Pure CPU Thread-Safe Resize
        # cv2.INTER_AREA is specifically designed for shrinking images cleanly
        mel_resized = cv2.resize(mel_db, (128, 128), interpolation=cv2.INTER_AREA)
        
        # 4. Add the channel dimension for the CNN (128, 128, 1)
        mel_expanded = np.expand_dims(mel_resized, axis=-1) 
        
        label_text = f"{machine}_{state}"
        return {"mel": mel_expanded, "label": LABEL_MAP[label_text]}
    except Exception:
        return None

## Data Loading and Mel-Spec Extraction

In [ ]:
tasks = []
task_labels = []

for machine in ["Machine 1", "Machine 2", "Machine 3"]:
    for state in ["Normal", "Abnormal"]:
        folder = os.path.join(BASE_PATH, machine, 'machine_data', state)
        if os.path.exists(folder):
            for file in os.listdir(folder):
                if file.endswith('.wav'):
                    tasks.append((machine, state, os.path.join(folder, file)))
                    task_labels.append(f"{machine}_{state}")

# --- THE 20% REDUCTION BLOCK ---
# print(f"Total files found: {len(tasks)}. Reducing to exactly 10%...")
# tasks_20, _, _, _ = train_test_split(
#     tasks, task_labels, train_size=0.10, random_state=42, stratify=task_labels
# )

results = []
print(f"⚙️ CPU Parallel Extraction of {len(tasks)} files...")
with ProcessPoolExecutor(max_workers=None) as executor:
    # Notice we are using tasks_20 here, not tasks
    futures = [executor.submit(extract_standardized_mel, task) for task in tasks]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Spectrograms"):
        res = future.result()
        if res is not None:
            results.append(res)

In [ ]:
import gc

print("📦 Converting to numpy arrays...")
# Use float32 for X and int8 for y to save massive amounts of RAM
X_raw = np.array([r["mel"] for r in results], dtype=np.float32)
y_full = np.array([r["label"] for r in results], dtype=np.int8)

# Nuke the original dictionary list from memory immediately
del results
gc.collect()

print("⚖️ Performing memory-safe MinMax Scaling...")
# Bypass sklearn completely. Do global MinMax mathematically to prevent RAM spikes.
X_min = X_raw.min()
X_max = X_raw.max()
X_scaled = (X_raw - X_min) / (X_max - X_min)

# Nuke X_raw to free up another ~3.6GB
del X_raw
gc.collect()

print(f"✅ Data Ready! Final Shape: {X_scaled.shape}")

# ─── SAVE CHECKPOINT ─────────────────────────────────────────────────────────
print(f"💾 Saving to disk at {SAVE_PATH}...")
# This will result in a file around 1.5 - 2.5 GB (well under the 19.5GB limit)
np.savez_compressed(SAVE_PATH, X=X_scaled, y=y_full)
print("🎉 Checkpoint saved successfully! You survived the preprocessing!")

## init kaggle dataset

In [ ]:
# import os

# # 1. Create a dedicated folder and move the file there
# os.makedirs('/kaggle/working/my_new_dataset', exist_ok=True)
# !mv /kaggle/working/cleaned_melspecs_checkpoint.npz /kaggle/working/my_new_dataset/

# # 2. Initialize the dataset metadata
# !kaggle datasets init -p /kaggle/working/my_new_dataset

# # 3. Modify the metadata file with a unique name
# import json
# meta_path = '/kaggle/working/my_new_dataset/dataset-metadata.json'
# with open(meta_path, 'r') as f:
#     meta = json.load(f)

# # Change these to your actual Kaggle username and a unique dataset name
# meta['id'] = 'YOUR_KAGGLE_USERNAME/cmp27-cleaned-melspecs-v1'
# meta['title'] = 'CMP27 Cleaned Melspecs Checkpoint'

# with open(meta_path, 'w') as f:
#     json.dump(meta, f)

# # 4. Push it to Kaggle!
# !kaggle datasets create -p /kaggle/working/my_new_dataset --dir-mode zip